# 08 · Serve a model and call it: the API and streaming
Level L1 · Part 1 · core · ~25 min · builds on 07

Time: about 25 min of work. Step 2b adds a **~6 min wait** on a 4-core CPU while the server warms up. Spend that wait reading the Picture and The idea. The notebook is not frozen.

Lesson 07 timed a stand-in. This lesson starts a **real model server**: the pinned vLLM CPU image (`vllm/vllm-openai-cpu:v0.32.0`) in Docker, serving `Qwen/Qwen2.5-0.5B-Instruct` (Apache-2.0, about 1 GB). It runs on a laptop with no account and no GPU. This lab server uses its own port **8081** and ignores `VLLM_BASE_URL` in `.env`: that value points at port 8000, the default port of a vLLM server, not this lab's port.

Without Docker, every cell that needs the server prints **recorded output** from the same image, labelled as such, so the rest of the lesson still reads correctly.

## Before you start
You need Docker running, about 1.2 GB of disk for the image and 1 GB for the model (downloaded once, then cached). You need about **5 GB of free RAM**: the server loads the weights in float32 (0.49 billion weights × 4 bytes ≈ 2 GB), reserves a 2 GiB cache for the attention keys and values (the KV cache, lesson 03), and the engine itself takes about 1 GB more. The container also gets 4 GB of shared memory for the engine's worker queues. Nothing breaks if Docker is missing: the server cells use the recorded output.

## Goal
By the end of this lesson you can start an OpenAI-compatible server, call it for a whole reply and for a stream of tokens, read the raw streamed lines, see a dropped stream free its request on the server, and keep the API key out of command lines and output.

## Why it matters
- **Without a server:** every application loads the weights itself. Each copy of a 1 GB model takes memory and load time, and each copy answers one request at a time. Every engine also has its own request format, so client code is written again for each one.
- **With a server:** one process holds the weights once, runs several requests in a batch, and speaks the OpenAI request format. The same client code works with vLLM, SGLang or Ollama (lesson 15 compares them). Streaming sends each token as soon as it exists, so the user reads while the model is still writing.
- **When you don't need it:** a script that answers one question a day can load the model itself. You need a server as soon as several callers share one model.

The cell below puts numbers on the first point.

In [1]:
params = 0.49e9                      # Qwen2.5-0.5B: about 0.49 billion weights
weights_gb = params * 4 / 1e9        # float32 stores 4 bytes per weight; this CPU server runs float32
apps = 4
print(f"one copy of the weights (float32): {weights_gb:.2f} GB")
print(f"{apps} apps, each with its own copy: {apps * weights_gb:.2f} GB")
print(f"one shared server: {weights_gb:.2f} GB, batching up to 4 requests (illustrative)")
print("on a GPU, bf16 stores 2 bytes per weight and halves these numbers")

one copy of the weights (float32): 1.96 GB
4 apps, each with its own copy: 7.84 GB
one shared server: 1.96 GB, batching up to 4 requests (illustrative)
on a GPU, bf16 stores 2 bytes per weight and halves these numbers


## The idea
An **OpenAI-compatible server** is a web server with the same routes and JSON shapes as OpenAI's API. `/v1/models` lists the models. `/v1/chat/completions` answers a conversation. The OpenAI Python SDK only needs a `base_url` to reach it, so your application code stays the same when the engine changes.

**Streaming** (`stream=true`) sends the answer as **server-sent events (SSE)**: one long HTTP response, with each new token written as a `data:` line as soon as the model produces it. The client reads at its own pace. If it reads slowly, the bytes wait in network buffers. That wait is **backpressure**. If the client closes the stream, the server should stop the request so that its compute and cache are freed. Step 5b checks that claim on the real server.

Think of a kitchen hatch: the cook can only pass plates as fast as the waiter takes them.

## Picture
```mermaid
flowchart LR
  A["Your app: OpenAI SDK with base_url"] -->|"POST /v1/chat/completions"| S["vLLM server on port 8081"]
  S --> E["Engine: batching and KV cache"]
  E -->|"data: token, data: token, ... [DONE]"| A
```

## Step by step
### Step 1: settings and helpers
The server key, if you set one, comes from `.env` through `env()`. The placeholder in `.env.example` counts as not set, so this lesson runs with no key. The key is never printed: the cell says only whether one is set. The cell also creates the Hugging Face cache folder first: if Docker creates it, it is owned by root.

In [2]:
import sys, os, re, json, time, socket, atexit, subprocess
from pathlib import Path
import httpx
sys.path.insert(0, "..")          # the course folder, so tut_env.py can be imported
from tut_env import env

IMAGE = "vllm/vllm-openai-cpu:v0.32.0"   # pinned; the same image that lessons 11, 12, 13, 14 and 15 probe (only this lesson starts a server from it)
NAME = "tut-kv-forge-15h-08"             # container name, unique to this course and lesson
PORT = 8081                              # this lesson's port
BASE = f"http://127.0.0.1:{PORT}"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
SERVED = "brightside-qwen-0.5b"          # the name clients send
KEY = env("VLLM_API_KEY", "")            # empty when unset: this lab server needs no key
AUTH = {"Authorization": f"Bearer {KEY}"} if KEY else {}
# The policy is made up for the Brightside example; the model can only answer from it.
SYSTEM = ("You are the Brightside support desk. Answer in one or two short sentences, using only this policy. "
          "Policy: opened items can be returned within 30 days with the receipt. Refunds reach the card in 5 business days.")
MSGS = [{"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Can I return an opened item?"}]
Path("../data/08").mkdir(parents=True, exist_ok=True)
Path.home().joinpath(".cache", "huggingface").mkdir(parents=True, exist_ok=True)
print("port", PORT, "| API key set:", bool(KEY))

port 8081 | API key set: False


Two helpers. `server_up()` asks the health route. `recorded()` reads the saved output from a real run, for the cells that run without a server; it prints the banner once per cell.

In [3]:
def server_up():
    try:
        return httpx.get(f"{BASE}/health", timeout=2).status_code == 200
    except httpx.TransportError:
        return False

def recorded(*keys):
    """Values saved from a real run on the pinned image (data/recorded/08.json)."""
    rec = json.loads(Path("../data/recorded/08.json").read_text())
    print(rec["note"])
    return [rec[k] for k in keys]

print("server answering:", server_up())

server answering: False


### Step 2: start the server and wait for it
The cell starts the container in the background. The key goes into the container's environment, not the command line, so no process list shows it.

The flags, one line each: `--dtype float32` runs on any x86 CPU (bf16 needs CPU bf16 units); `--enforce-eager` skips the one-time `torch.compile`; `--max-num-seqs 4` allows at most 4 requests in one batch; `--kv-cache-memory-bytes` reserves 2 GiB for the KV cache. Lesson 16 explains these flags. `--shm-size 4g` matters: Docker's 64 MB default is too small, and the start fails with a message about `/dev/shm`.

In [4]:
with socket.socket() as s:
    busy = s.connect_ex(("127.0.0.1", PORT)) == 0
if busy:
    raise RuntimeError(f"port {PORT} is busy: stop the other process first")
subprocess.run(["docker", "rm", "-f", NAME], capture_output=True)   # clear a container left by an interrupted run
args = [MODEL, "--served-model-name", SERVED, "--host", "0.0.0.0", "--dtype", "float32", "--enforce-eager",
        "--max-model-len", "4096", "--max-num-seqs", "4", "--kv-cache-memory-bytes", str(2 * 1024**3)]
run_env, key_flag = dict(os.environ), []
if KEY:
    run_env["VLLM_API_KEY"] = KEY               # the server reads this variable itself
    key_flag = ["-e", "VLLM_API_KEY"]
cmd = ["docker", "run", "-d", "--name", NAME, "-p", f"{PORT}:8000", "--shm-size", "4g",
       "-v", f"{Path.home() / '.cache' / 'huggingface'}:/root/.cache/huggingface", *key_flag, IMAGE, *args]
started = subprocess.run(cmd, capture_output=True, text=True, env=run_env, timeout=120)
if started.returncode != 0:
    raise RuntimeError("docker run failed: " + started.stderr[-300:])
atexit.register(lambda: subprocess.run(["docker", "rm", "-f", NAME], capture_output=True))
print("container started:", NAME)

⏭  Skipped: this cell needs [docker,download]. Run it yourself, or re-run with TUT_ENABLE=docker,download.


The wait below takes about a minute with a warm cache, plus the model download on a first run.

In [5]:
deadline = time.time() + 480
while time.time() < deadline:
    state = subprocess.run(["docker", "inspect", "-f", "{{.State.Running}}", NAME],
                           capture_output=True, text=True).stdout.strip()
    if state != "true":
        logs = subprocess.run(["docker", "logs", "--tail", "20", NAME], capture_output=True, text=True)
        raise RuntimeError("server container stopped:\n" + (logs.stdout + logs.stderr)[-800:])
    if server_up():
        break
    time.sleep(2)
else:
    raise TimeoutError(f"server not healthy after 480 s on port {PORT}")
print("server healthy on port", PORT)

⏭  Skipped: this cell needs [docker,download]. Run it yourself, or re-run with TUT_ENABLE=docker,download.


### Step 2b: the first answer is slow on a CPU (~6 min wait here)
`/health` answers as soon as the server is up. The first request is different: on a CPU the engine compiles and sets up its math kernels on the first request only. On this 4-core CPU that takes about **6 minutes**. Later requests take seconds. A GPU does the same setup in seconds.

The notebook is not frozen, so do not interrupt it. This cell sends one tiny request and times it, so the wait is visible and the next cells are not surprised by it. If it times out, re-run the cell: the warm-up usually finishes on the server anyway.

In [6]:
if server_up():
    print("sending the warm-up request, this can take about 6 minutes...")
    t0 = time.perf_counter()
    try:
        first = httpx.post(f"{BASE}/v1/chat/completions", headers=AUTH, timeout=540,
                           json={"model": SERVED, "messages": [{"role": "user", "content": "Hi"}],
                                 "max_tokens": 1, "temperature": 0})
        print(f"first request: HTTP {first.status_code} after {time.perf_counter() - t0:.0f} s (one-time CPU warm-up)")
    except httpx.TimeoutException:
        print("timed out at 540 s: re-run this cell; the warm-up usually finishes on the server anyway")
else:
    (first,) = recorded("first_request")
    print("first request (recorded):", first)

⏭  Skipped: this cell needs [docker,download]. Run it yourself, or re-run with TUT_ENABLE=docker,download.


### Step 3: list the models and ask one question
The OpenAI SDK sends the same request it would send to OpenAI. The model name must be the one `/v1/models` lists. `temperature=0` makes the answer repeatable. The `base_url` ends in `/v1`, because the routes live under it.

In [7]:
from openai import OpenAI
client = OpenAI(base_url=f"{BASE}/v1", api_key=KEY or "not-needed")   # a local server ignores the key when none is set
if server_up():
    print("models:", [m.id for m in client.models.list().data])
    resp = client.chat.completions.create(model=SERVED, messages=MSGS, max_tokens=60, temperature=0)
    print("reply:", resp.choices[0].message.content)
    print("tokens in / out:", resp.usage.prompt_tokens, "/", resp.usage.completion_tokens)
else:
    models, reply, usage = recorded("models", "reply", "usage")
    print("models:", models)
    print("reply:", reply)
    print("tokens in / out:", usage)

recorded output: a live run of this notebook on the course machine (Docker, pinned CPU image, 4 CPU cores, no GPU), 2026-10-22
models: ['brightside-qwen-0.5b']
reply: Yes, you can return an opened item within 30 days of receiving it with a valid receipt.
tokens in / out: 68 / 22


**What you should see:** the model list shows `brightside-qwen-0.5b`. The reply uses the policy line in the system message: the 30 days and the receipt. The prompt is a few dozen tokens, and the answer is well under the 60-token limit.

### Step 4: the same question, streamed
With `stream=True` the SDK returns chunks as they arrive. The first chunk with text ends the wait for the first token. The last chunk carries the usage counts because the request asked for `include_usage`.

In [8]:
if server_up():
    t0 = time.perf_counter()
    stream = client.chat.completions.create(model=SERVED, messages=MSGS, max_tokens=60, temperature=0,
                                            stream=True, stream_options={"include_usage": True})
    pieces, first, usage = [], None, None
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            if first is None:
                first = time.perf_counter() - t0
            pieces.append(chunk.choices[0].delta.content)
        if chunk.usage:
            usage = chunk.usage
    print(f"first token after {first * 1000:.0f} ms, {len(pieces)} text chunks, {usage.completion_tokens} tokens")
    print("".join(pieces))
else:
    first_ms, chunks, text = recorded("stream_first_ms", "stream_chunks", "stream_text")
    print("first token after", first_ms, "ms,", chunks, "text chunks")
    print(text)

recorded output: a live run of this notebook on the course machine (Docker, pinned CPU image, 4 CPU cores, no GPU), 2026-10-22
first token after 1503 ms, 21 text chunks
Yes, you can return an opened item within 30 days of receiving it with a valid receipt.


The first token is fast because the warm-up in step 2b is already done. Before that warm-up, this first token would have taken minutes.

### Step 5: the wire format
Each line of the raw response is one SSE event. The cell prints the first lines, one token each, and the last line. The first line is cut short. The stream ends with `data: [DONE]`, the last line.

In [9]:
def show_sse(lines):
    """Print the first line, the next two token deltas, and the last line."""
    print(lines[0][:80])
    for l in lines[1:3]:
        print(json.loads(l[6:])["choices"][0]["delta"])     # l[6:] drops the leading "data: "
    print(lines[-1])

if server_up():
    body = {"model": SERVED, "messages": MSGS, "max_tokens": 8, "temperature": 0, "stream": True}
    with httpx.stream("POST", f"{BASE}/v1/chat/completions", json=body, headers=AUTH, timeout=60) as r:
        lines = [l for l in r.iter_lines() if l]           # the whole stream, to the end
    Path("../data/08/sse_lines.json").write_text(json.dumps(lines))   # saved for the recorded output
    show_sse(lines)
else:
    (lines,) = recorded("sse_lines")
    print("raw lines (recorded):")
    show_sse(lines)

recorded output: a live run of this notebook on the course machine (Docker, pinned CPU image, 4 CPU cores, no GPU), 2026-10-22
raw lines (recorded):
data: {"id":"chatcmpl-987141e85e152e3d","object":"chat.completion.chunk","create
{'content': 'Yes'}
{'content': ','}
data: [DONE]


### Step 5b: a dropped stream
Now the test that matters for cost. The client reads five lines and leaves. The server should stop the request. The cell reads the running-request gauge from `/metrics` while the stream is still open, then times how long it takes to fall to zero after the client has gone.

In [10]:
def running():
    text = httpx.get(f"{BASE}/metrics", timeout=5).text
    hit = re.search(r"^vllm:num_requests_running\{[^}]*\}\s+(\S+)", text, re.M)
    return float(hit.group(1)) if hit else None

if server_up():
    body = {"model": SERVED, "messages": MSGS, "max_tokens": 400, "temperature": 0, "stream": True}
    with httpx.stream("POST", f"{BASE}/v1/chat/completions", json=body, headers=AUTH, timeout=60) as r:
        for i, _ in enumerate(r.iter_lines()):
            if i == 5:
                break                                # read five lines, then stop reading
        before = running()                           # the stream is still open here
    # the with-block ends above: the client closes the stream and has left
    t_close = time.perf_counter()
    after = before
    while after not in (0.0, None) and time.perf_counter() - t_close < 20:
        time.sleep(0.2)
        after = running()
    print(f"running before the client left: {before}; after: {after} (waited {time.perf_counter() - t_close:.1f} s)")
else:
    before, after, secs = recorded("running_before", "running_after", "abort_seconds")
    print("running before:", before, "| after:", after, "| waited:", secs, "s")

recorded output: a live run of this notebook on the course machine (Docker, pinned CPU image, 4 CPU cores, no GPU), 2026-10-22
running before: 1.0 | after: 0.0 | waited: 0.5 s


**What you should see:** the count is 1 before the client leaves and 0 a moment later. The server stopped the request, so it spends no more compute on it. If the count stayed at 1, the server would keep generating for nobody, and the proxy in front of it would have to close the upstream connection (lesson 30 covers routing, lesson 33 the failure drills).

### Step 6: the key, and what stays out of the output
With a key set, the server answers 401 to a call without it. The cell prints only status codes and whether a key is set. It never prints the key. The recording was made without a key, so the 401 pair is **not** in the recorded output: set `VLLM_API_KEY` in `.env` and re-run to see it live.

In [11]:
if server_up() and KEY:
    print("call without the key:", httpx.get(f"{BASE}/v1/models", timeout=5).status_code)
    print("call with the key:", httpx.get(f"{BASE}/v1/models", headers=AUTH, timeout=5).status_code)
elif server_up():
    print("no VLLM_API_KEY in .env, so this lab server takes calls without one.")
    print("Set it in .env and run again to see the 401 path. The key is never printed.")
else:
    (note,) = recorded("key_note")
    print("recorded (no key was set when recorded):", note)

recorded output: a live run of this notebook on the course machine (Docker, pinned CPU image, 4 CPU cores, no GPU), 2026-10-22
recorded (no key was set when recorded): no VLLM_API_KEY was set when recorded, so the lab server took calls without a key


## At work
- **Dropped streams cost money.** A proxy that buffers SSE, or does not close the upstream connection, keeps the engine generating for a client that has left. Check that the running count falls after a disconnect, as in step 5b. → lesson 30 (routing) and lesson 33 (failure drills).
- **A gateway can send a request to the wrong pool.** Anthropic's 2025 postmortem traced a routing change that sent short requests to servers built for a much longer window. Log the model and the route with every request. → lesson 30.
- **Streams cannot move between servers.** When a node dies mid-answer, the stream is gone and the client must retry; the partial text can be sent back as a prefix. → lesson 33.

Sources: vLLM docs, OpenAI-compatible server (https://docs.vllm.ai/en/latest/serving/online_serving/); Anthropic, "A postmortem of three recent issues" (https://www.anthropic.com/engineering/a-postmortem-of-three-recent-issues).

## Common mistakes
**1. A wrong model name.** The server only knows the name it serves (`brightside-qwen-0.5b`). A name from another provider gets a 404. The cell below sends one and prints the real error, then lists the names the server does serve.

In [12]:
if server_up():
    try:
        client.chat.completions.create(model="gpt-4o", messages=MSGS, max_tokens=5)
    except Exception as exc:
        print(type(exc).__name__, "-", str(exc)[:120])
    print("served names:", [m.id for m in client.models.list().data])
else:
    err, names = recorded("wrong_model_error", "served_names")
    print("recorded:", err)
    print("served names (recorded):", names)

recorded output: a live run of this notebook on the course machine (Docker, pinned CPU image, 4 CPU cores, no GPU), 2026-10-22
recorded: NotFoundError - Error code: 404 - {'error': {'message': 'The model `gpt-4o` does not exist. Valid aliases: brightside-qwen-0.5b.', 'type
served names (recorded): ['brightside-qwen-0.5b']


**The fix:** read the names from `client.models.list()` and send one of them.

**2. A `base_url` without `/v1`.** The routes live under `/v1`, so the call goes to a path that does not exist. The cell below uses the wrong `base_url` and prints the real error.

In [13]:
if server_up():
    try:
        OpenAI(base_url=BASE, api_key=KEY or "not-needed").chat.completions.create(model=SERVED, messages=MSGS, max_tokens=5)
    except Exception as exc:
        print(type(exc).__name__, "-", str(exc)[:120])
else:
    (err,) = recorded("base_url_error")
    print("recorded:", err)

recorded output: a live run of this notebook on the course machine (Docker, pinned CPU image, 4 CPU cores, no GPU), 2026-10-22
recorded: NotFoundError - Error code: 404 - {'detail': 'Not Found'}


**The fix:** end `base_url` with `/v1`, as step 3 does. The Try-it below uses the same client.

## Try it
Stream a reply to a new question, "How long do refunds take?", with `max_tokens=30`. Print the number of text chunks and the first 60 characters of the reply. The server from step 2 must still be running.

In [14]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [15]:
if server_up():
    stream = client.chat.completions.create(model=SERVED, max_tokens=30, temperature=0, stream=True,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": "How long do refunds take?"}])
    text = [c.choices[0].delta.content for c in stream if c.choices and c.choices[0].delta.content]
    print(len(text), "text chunks;", "".join(text)[:60])
else:
    (answer,) = recorded("try_it")
    print("recorded:", answer)

recorded output: a live run of this notebook on the course machine (Docker, pinned CPU image, 4 CPU cores, no GPU), 2026-10-22
recorded: 12 text chunks; Refunds typically reach your card within 5 business days.


## Recap
- An OpenAI-compatible server speaks the OpenAI routes and JSON, so one SDK call works for any engine that does.
- `stream=true` sends each token as an SSE `data:` line; the stream ends with `[DONE]`.
- Backpressure is the wait that builds when a client reads slowly. A dropped stream should free its request on the server; measure that, do not assume it.
- Keep the key in the environment and out of command lines and output.

**Interview:** "How would you expose a self-hosted model to other teams?" Strong answer: an OpenAI-compatible server behind a gateway; streaming for interactive use; the gateway closes upstream connections on disconnect; keys stay in the environment and out of logs.

## Clean up
This is the last cell. It saves the last lines of the server log to `data/08/` and removes the container. It also runs when the server never started.

In [16]:
try:
    logs = subprocess.run(["docker", "logs", "--tail", "200", NAME], capture_output=True, text=True)
    Path("../data/08/server.log").write_text(logs.stdout + logs.stderr)
except FileNotFoundError:
    pass
subprocess.run(["docker", "rm", "-f", NAME], capture_output=True)
print("container removed; server log saved to data/08/server.log")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.
